# 09 V6+V10 ensemble + TTA — submission-gain check (eval only)

**Evaluation only.** Attacks the score from the *all-class / zero-training* side after the small-object
lines (two-stage, MedSAM, NWD) closed. Two near-tied diverse models (V6, V10 ≈ 0.234) are merged by
**class-wise NMS**, with **manual horizontal-flip TTA** (Ultralytics `augment=True` is a no-op for seg
models — it warns and reverts to single-scale — so TTA is done by hand: predict on the flipped image,
mirror detections back, merge). Scored on the FULL-image val set with the **same self-contained Mask
mAP code as src/03/04/05**, so the ensemble/TTA gain is a true delta — not a metric artifact — vs
single-model V6 (re-scored **0.2099**).

**Why eval-first:** instance-seg ensembling does not always help; a blind submission wastes a slot
and hides whether the gain is real. This notebook proves the gain on val before any submission. If it
clears V6 beyond the ~0.003 noise band (and large classes hold), wire the winning config into a
submission via `src/02`.

**Variants scored (full attribution):** `V6`, `V10`, `V6+TTA`, `V10+TTA`, `Ensemble`, `Ensemble+TTA`.
The headline is **`Ensemble+TTA` vs `V6` (this notebook's own anchor, which should land near 0.2099)**.

**Inputs (add as Kaggle Datasets/Models):** the **V6** detector (`version6...best.pt`) and the **V10**
detector (`version10...best.pt`); auto-detected by filename. No training, no extra weights.

**Decisions baked in (chosen with the user):** eval-only; class-wise **NMS** merge; **manual hflip** TTA.

## 1. Setup

In [ ]:
try:
    import ultralytics  # noqa
    print("ultralytics present:", ultralytics.__version__)
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ultralytics"])

In [ ]:
import os, json
from pathlib import Path
import numpy as np
import cv2
import yaml
import torch
import pandas as pd
from ultralytics import YOLO

cv2.setNumThreads(0)
SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

## 2. Configuration

`RUN_NOTTA_ANCHOR` keeps the single-model / no-TTA rows so the gain is fully attributable
(TTA alone vs ensemble alone vs both). Set it `False` to halve inference time once you trust the
anchor. The cross-model merge uses class-wise NMS at `ENS_NMS_IOU`.

In [ ]:
# --- What to run ---
RUN_NOTTA_ANCHOR = True     # include single-model / ensemble at no TTA (attribution + 0.2099 anchor)
RUN_TTA          = True     # add manual horizontal-flip (hflip) TTA variants
# NOTE: Ultralytics `augment=True` is a no-op for SEG models (it warns + reverts to single-scale),
# so TTA here is done MANUALLY: predict on the hflipped image, mirror masks/boxes back, then merge.

# --- Detector inference ---
IMG_SIZE      = 768         # V6 and V10 were both trained at 768
CAPTURE_CONF  = 0.001       # capture low so mask-AP sees the full PR curve (do NOT hard-threshold for AP)
PRED_NMS_IOU  = 0.70        # each model's own predict-time NMS IoU (YOLO default)
MAX_DET       = 300

# --- Ensemble / TTA merge (class-wise NMS over pooled detections) ---
ENS_NMS_IOU   = 0.60        # box-IoU above which two pooled detections are treated as the same object

# --- Mask-AP (identical to src/03/04/05) ---
IOU_THRESHOLDS = np.linspace(0.5, 0.95, 10)

# V6 per-class Mask mAP50-95 reference (src/03 comparable metric) for context.
V6_REF_AP = {"abrasion":0.6471,"crown":0.6313,"filling":0.2799,"caries 1":0.1195,
             "caries 2":0.0845,"caries 3":0.0116,"caries 4":0.0000,"caries 5":0.1097,"caries 6":0.0051}
V6_REF_MAP = 0.2099

def _norm_class_key(nm):
    s = str(nm).lower().replace("class", "")
    return " ".join(s.split())

assert RUN_NOTTA_ANCHOR or RUN_TTA, "Enable at least one of RUN_NOTTA_ANCHOR / RUN_TTA."
VIEWS = ["orig"] + (["hflip"] if RUN_TTA else [])   # per-model inference views; hflip is the manual TTA
print("views:", VIEWS, "| imgsz", IMG_SIZE, "| capture conf", CAPTURE_CONF, "| ens NMS IoU", ENS_NMS_IOU)

## 3. Dataset + validation ground truth

In [ ]:
INPUT_ROOT = Path("/kaggle/input")
yc = list(INPUT_ROOT.rglob("yolo_seg_train.yaml"))
if not yc:
    raise FileNotFoundError("No yolo_seg_train.yaml under /kaggle/input.")
DATA_YAML = yc[0]
dcfg = yaml.safe_load(open(DATA_YAML, encoding="utf-8"))
names = dcfg.get("names")
if isinstance(names, dict):
    names = [names[k] for k in sorted(names)]
num_classes = len(names)
dataset_root = DATA_YAML.parent
yaml_root = Path(dcfg["path"]) if dcfg.get("path") else dataset_root
if not yaml_root.is_absolute():
    yaml_root = dataset_root / yaml_root

def resolve_split(v):
    if v is None: return None
    p = Path(v)
    if p.is_absolute(): return p
    for c in (yaml_root / p, dataset_root / p):
        if c.exists(): return c
    return yaml_root / p

val_images = resolve_split(dcfg.get("val", dcfg.get("valid")))
def labels_dir_for(images_dir):
    parts = list(Path(images_dir).parts)
    if "images" in parts:
        i = len(parts) - 1 - parts[::-1].index("images"); parts[i] = "labels"
        return Path(*parts)
    return Path(images_dir).parent / "labels"

IMG_EXTS = {".jpg",".jpeg",".png",".bmp",".webp",".tif",".tiff"}
def parse_seg_line(line):
    parts = line.strip().split()
    if len(parts) < 7: return None
    try:
        cls = int(float(parts[0])); coords = [float(v) for v in parts[1:]]
    except ValueError:
        return None
    if len(coords) % 2: coords = coords[:-1]
    poly = np.asarray(coords, dtype=np.float64).reshape(-1, 2)
    return (cls, poly) if len(poly) >= 3 else None

val_objs = {}
lbl_dir = labels_dir_for(val_images)
for ip in sorted(p for p in Path(val_images).iterdir() if p.suffix.lower() in IMG_EXTS):
    lp = lbl_dir / (ip.stem + ".txt")
    objs = []
    if lp.exists():
        for line in lp.read_text().splitlines():
            pr = parse_seg_line(line)
            if pr is not None: objs.append(pr)
    val_objs[str(ip)] = objs

n_gt = np.zeros(num_classes, dtype=int)
for objs in val_objs.values():
    for c, _ in objs: n_gt[c] += 1
print("classes:", names)
print("val images:", len(val_objs), "| total GT objects:", int(n_gt.sum()))
for c in range(num_classes):
    print(f"  {names[c]:>14s}: n_gt={int(n_gt[c])}")

## 4. Load the V6 and V10 detectors (auto-detect by filename)

In [ ]:
MANUAL_V6_PATH  = None    # set to exact /kaggle/input path if auto-detect picks wrong
MANUAL_V10_PATH = None

def find_pt(include, exclude=()):
    cands = []
    for p in Path("/kaggle/input").rglob("*.pt"):
        t = str(p).lower()
        if any(x in t for x in exclude): continue
        score = sum(10 for k in include if k in t) + (20 if p.name.lower().endswith("best.pt") else 0)
        if score > 0: cands.append((score, p))
    cands.sort(key=lambda z: z[0], reverse=True)
    return [p for _, p in cands]

v6  = find_pt(["version6", "v6"],   exclude=["stage2", "version10", "version13", "version14", "version15"])
v10 = find_pt(["version10", "v10"], exclude=["stage2", "version6"])
V6_PATH  = Path(MANUAL_V6_PATH)  if MANUAL_V6_PATH  else (v6[0]  if v6  else None)
V10_PATH = Path(MANUAL_V10_PATH) if MANUAL_V10_PATH else (v10[0] if v10 else None)
print("V6 :", V6_PATH)
print("V10:", V10_PATH)
if V6_PATH is None or not V6_PATH.exists():
    raise FileNotFoundError("No V6 detector found. Add version6_...best.pt or set MANUAL_V6_PATH.")
if V10_PATH is None or not V10_PATH.exists():
    raise FileNotFoundError("No V10 detector found. Add version10_...best.pt or set MANUAL_V10_PATH.")

detectors = {"V6": YOLO(str(V6_PATH)), "V10": YOLO(str(V10_PATH))}
# (No augment=True probe: Ultralytics seg ignores it. TTA is the manual hflip pass in section 5.)

## 5. Capture predictions — original + manual hflip (TTA)

Per model: one pass on the original image, and (if `RUN_TTA`) one pass on the horizontally-flipped
image whose detections are mirrored back to the original frame (poly `x→1−x`, box x mirrored). Low
conf so mask-AP sees the full PR curve. Stored as `preds[(model, view)][img]`, `view ∈ {orig, hflip}`.

In [ ]:
def predict_dets(det, img):
    res = det.predict(source=img, imgsz=IMG_SIZE, conf=CAPTURE_CONF, iou=PRED_NMS_IOU,
                      max_det=MAX_DET, device=DEVICE, task="segment", verbose=False, save=False)[0]
    dets = []
    if res.masks is not None and res.boxes is not None and len(res.boxes) > 0:
        xyxy = res.boxes.xyxy.cpu().numpy()
        conf = res.boxes.conf.cpu().numpy()
        cls  = res.boxes.cls.cpu().numpy().astype(int)
        xyn  = res.masks.xyn
        for i in range(min(len(xyxy), len(xyn))):
            poly = np.asarray(xyn[i], dtype=np.float64)
            if poly.ndim != 2 or len(poly) < 3: continue
            x0, y0, x1, y1 = [float(v) for v in xyxy[i]]
            dets.append({"cls": int(cls[i]), "conf": float(conf[i]),
                         "box": (x0, y0, x1, y1), "poly": poly})
    return dets

def hflip_back(dets, w):
    # Mirror detections from an hflipped image back to the original frame.
    out = []
    for d in dets:
        x0, y0, x1, y1 = d["box"]
        p = d["poly"].copy(); p[:, 0] = 1.0 - p[:, 0]
        out.append({"cls": d["cls"], "conf": d["conf"], "box": (w - x1, y0, w - x0, y1), "poly": p})
    return out

preds = {(n, v): {} for n in detectors for v in VIEWS}
wh_by_image = {}
for ip in val_objs:
    img = cv2.imread(ip, cv2.IMREAD_COLOR)
    h, w = img.shape[:2]; wh_by_image[ip] = (w, h)
    img_f = cv2.flip(img, 1) if "hflip" in VIEWS else None
    for name, det in detectors.items():
        preds[(name, "orig")][ip] = predict_dets(det, img)
        if "hflip" in VIEWS:
            preds[(name, "hflip")][ip] = hflip_back(predict_dets(det, img_f), w)
for k in preds:
    print(f"{k}: {sum(len(v) for v in preds[k].values())} detections")

## 6. Class-wise NMS + variant detection sets

`nms_merge` runs greedy class-wise NMS (keep highest-conf, drop box-IoU ≥ `ENS_NMS_IOU`). A `+TTA`
single-model variant = its `orig` + `hflip` detections merged; the ensemble pools both models'
views and merges. No-TTA single-model variants use the native `orig` output directly.

In [ ]:
def box_iou(a, b):
    ix0, iy0 = max(a[0], b[0]), max(a[1], b[1])
    ix1, iy1 = min(a[2], b[2]), min(a[3], b[3])
    iw, ih = max(0.0, ix1 - ix0), max(0.0, iy1 - iy0)
    inter = iw * ih
    if inter <= 0: return 0.0
    aa = (a[2]-a[0])*(a[3]-a[1]); bb = (b[2]-b[0])*(b[3]-b[1])
    return inter / (aa + bb - inter)

def nms_merge(dets, iou_thr):
    keep = []
    for c in set(d["cls"] for d in dets):
        cd = sorted([d for d in dets if d["cls"] == c], key=lambda x: x["conf"], reverse=True)
        sel = []
        for d in cd:
            if all(box_iou(d["box"], s["box"]) < iou_thr for s in sel):
                sel.append(d)
        keep.extend(sel)
    return keep

def merge_views(view_keys, iou_thr):
    # view_keys: list of (model, view) keys in `preds`. Pool per image, then class-wise NMS.
    out = {}
    for ip in val_objs:
        pool = []
        for k in view_keys: pool.extend(preds[k].get(ip, []))
        out[ip] = nms_merge(pool, iou_thr)
    return out

VARIANTS = {}
if RUN_NOTTA_ANCHOR:
    VARIANTS["V6"]       = preds[("V6", "orig")]
    VARIANTS["V10"]      = preds[("V10", "orig")]
    VARIANTS["Ensemble"] = merge_views([("V6", "orig"), ("V10", "orig")], ENS_NMS_IOU)
if RUN_TTA:
    VARIANTS["V6+TTA"]       = merge_views([("V6", "orig"), ("V6", "hflip")], ENS_NMS_IOU)
    VARIANTS["V10+TTA"]      = merge_views([("V10", "orig"), ("V10", "hflip")], ENS_NMS_IOU)
    VARIANTS["Ensemble+TTA"] = merge_views([("V6", "orig"), ("V6", "hflip"),
                                            ("V10", "orig"), ("V10", "hflip")], ENS_NMS_IOU)
print("variants:", list(VARIANTS))

## 7. Score every variant with the comparable full-image Mask mAP

Mask-IoU matching (local-frame, exact), greedy conf-ranked, 10 IoU thresholds 0.5:0.95, 101-pt AP —
identical to src/03/04/05. Aggregate = mean over classes with support.

In [ ]:
def poly_to_local_mask(poly_norm, w, h):
    pts = poly_norm.copy(); pts[:, 0] *= w; pts[:, 1] *= h
    gx0 = max(0, int(np.floor(pts[:,0].min()))); gy0 = max(0, int(np.floor(pts[:,1].min())))
    gx1 = min(w, int(np.ceil(pts[:,0].max())));  gy1 = min(h, int(np.ceil(pts[:,1].max())))
    gw = max(1, gx1 - gx0); gh = max(1, gy1 - gy0)
    m = np.zeros((gh, gw), dtype=np.uint8)
    pts[:, 0] -= gx0; pts[:, 1] -= gy0
    cv2.fillPoly(m, [pts.astype(np.int32)], 1)
    return (gx0, gy0, gx1, gy1), m

def iou_local(pbox, pmask, gbox, gmask):
    pa = int(pmask.sum()); ga = int(gmask.sum())
    if pa == 0 or ga == 0: return 0.0
    ix0 = max(pbox[0], gbox[0]); iy0 = max(pbox[1], gbox[1])
    ix1 = min(pbox[2], gbox[2]); iy1 = min(pbox[3], gbox[3])
    inter = 0
    if ix1 > ix0 and iy1 > iy0:
        pc = pmask[iy0-pbox[1]:iy1-pbox[1], ix0-pbox[0]:ix1-pbox[0]]
        gc = gmask[iy0-gbox[1]:iy1-gbox[1], ix0-gbox[0]:ix1-gbox[0]]
        inter = int(np.logical_and(pc, gc).sum())
    union = pa + ga - inter
    return inter / union if union > 0 else 0.0

def ap_101(confs, tps, npos):
    if npos == 0: return float("nan")
    if not confs: return 0.0
    o = np.argsort(-np.asarray(confs)); tps = np.asarray(tps)[o]
    tp_c = np.cumsum(tps); fp_c = np.cumsum(1 - tps)
    rec = tp_c / npos; prec = tp_c / np.maximum(tp_c + fp_c, 1e-9)
    return sum((prec[rec >= r].max() if np.any(rec >= r) else 0.0) for r in np.linspace(0,1,101)) / 101.0

# GT local masks once.
gt_local = {ip: [(c, *poly_to_local_mask(p, *wh_by_image[ip])) for c, p in objs]
            for ip, objs in val_objs.items()}

def score_variant(dets_by_image):
    records = {(c, ti): [] for c in range(num_classes) for ti in range(len(IOU_THRESHOLDS))}
    for ip in val_objs:
        gts = gt_local[ip]
        pls = []
        for d in dets_by_image.get(ip, []):
            pbox, pm = poly_to_local_mask(d["poly"], *wh_by_image[ip])
            pls.append((d["cls"], d["conf"], pbox, pm))
        for ti, thr in enumerate(IOU_THRESHOLDS):
            order = sorted(range(len(pls)), key=lambda i: pls[i][1], reverse=True)
            used = [False] * len(gts)
            for pi in order:
                pc, sc, pbox, pm = pls[pi]
                best, bg = 0.0, -1
                for gi, (gc, gbox, gm) in enumerate(gts):
                    if used[gi] or gc != pc: continue
                    v = iou_local(pbox, pm, gbox, gm)
                    if v > best: best, bg = v, gi
                tp = 1 if (bg >= 0 and best >= thr) else 0
                if tp: used[bg] = True
                records[(pc, ti)].append((sc, tp))
    pac = np.full(num_classes, np.nan)
    for c in range(num_classes):
        if n_gt[c] == 0: continue
        pac[c] = np.nanmean([ap_101([r[0] for r in records[(c,ti)]],
                                    [r[1] for r in records[(c,ti)]], n_gt[c])
                             for ti in range(len(IOU_THRESHOLDS))])
    return pac

ap_by_variant = {tag: score_variant(dets) for tag, dets in VARIANTS.items()}
agg = {tag: float(np.nanmean(pac[~np.isnan(pac)])) for tag, pac in ap_by_variant.items()}

anchor = "V6" if "V6" in agg else ("V6+TTA" if "V6+TTA" in agg else list(agg)[0])
print(f"anchor = {anchor} (this notebook's V6, should land near documented 0.2099)\n")
print(f"{'variant':>16s}  {'mAP50-95':>9s}  {'vs anchor':>10s}  {'vs 0.2099':>10s}")
for tag in VARIANTS:
    print(f"{tag:>16s}  {agg[tag]:>9.4f}  {agg[tag]-agg[anchor]:>+10.4f}  {agg[tag]-V6_REF_MAP:>+10.4f}")

print("\nper-class Mask AP (rows = classes, cols = variants):")
hdr = "".join(f"{t:>14s}" for t in VARIANTS)
print(f"{'class':>14s}{hdr}")
for c in range(num_classes):
    row = "".join(f"{ap_by_variant[t][c]:>14.3f}" for t in VARIANTS)
    print(f"{names[c]:>14s}{row}")

## 8. Save outputs

In [ ]:
OUT = Path("/kaggle/working")
rows = []
for tag in VARIANTS:
    for c in range(num_classes):
        rows.append({"variant": tag, "class": names[c], "n_gt": int(n_gt[c]),
                     "AP": (None if np.isnan(ap_by_variant[tag][c]) else float(ap_by_variant[tag][c]))})
    rows.append({"variant": tag, "class": "__aggregate__", "n_gt": int(n_gt.sum()),
                 "AP": float(agg[tag])})
res_df = pd.DataFrame(rows)
res_csv = OUT / "ensemble_v6v10_tta_eval.csv"
res_df.to_csv(res_csv, index=False)

summary = {
    "config": {"IMG_SIZE": IMG_SIZE, "CAPTURE_CONF": CAPTURE_CONF, "ENS_NMS_IOU": ENS_NMS_IOU,
               "VIEWS": VIEWS, "tta": "manual hflip", "anchor": anchor},
    "aggregate_mAP": {t: round(agg[t], 4) for t in VARIANTS},
    "delta_vs_anchor": {t: round(agg[t] - agg[anchor], 4) for t in VARIANTS},
    "delta_vs_0.2099": {t: round(agg[t] - V6_REF_MAP, 4) for t in VARIANTS},
    "v6_ref_mAP": V6_REF_MAP,
}
with open(OUT / "ensemble_v6v10_tta_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("Saved to /kaggle/working:")
for p in ["ensemble_v6v10_tta_eval.csv", "ensemble_v6v10_tta_summary.json"]:
    fp = OUT / p
    print(f"  [{'OK' if fp.exists() else '--'}] {p}")
print("\n", json.dumps(summary["aggregate_mAP"], indent=2))

## 9. How to read this → submit or not

**Headline: `Ensemble+TTA` aggregate vs the `V6` anchor (and vs 0.2099).** Noise band ≈ 0.003.

- **`Ensemble+TTA` > anchor by > 0.003 AND large classes (Abrasion/Crown/Filling) don't regress**
  → real submission gain. Wire the winning config into `src/02`: for each model predict on the image
  **and its hflip** (mirror detections back), pool, class-wise NMS at `ENS_NMS_IOU`, emit polys.
- **Use the attribution rows** to keep only what pays: if `V6+TTA` ≈ `V6` but `Ensemble` helps, hflip
  TTA is dead weight (drop it, ensemble only); if `Ensemble` ≈ single but `+TTA` helps, skip the 2nd
  model. Pick the cheapest variant that captures the gain.
- **Sanity:** the `V6` anchor should sit near 0.2099. If it is far off, the val set / metric wiring
  differs from src/03 and the deltas — not the absolute — are still what matter.
- **Knobs if flat:** `ENS_NMS_IOU` (lower = merge more aggressively), confidence-weight the two models
  before pooling, or add more TTA views (vertical flip / multi-scale). Single-variable follow-ups, not
  reasons to abandon.

This is the comparable *dev* metric; both models are unchanged, so a confirmed val gain should
transfer to the leaderboard.